# Shaped Pulse Scheduler for TProc V2 (ZCU216 Version)
1-16 tone per channel pulse scheduler with pulse_shaping

## Initialization

In [ ]:
import os
from pathlib import Path

base = Path("/home/xilinx/jupyter_notebooks/qick/firmware/projects/qick_tprocv2_216_16mixmux16")
out_dir = base / "out"

# 1. Clean up the non-working links
for name in ["qick_216.bit", "qick_216.hwh"]:
    f = out_dir / name
    if f.exists() or f.is_symlink():
        f.unlink()

# 2. Create actual, native OS-level relative symlinks
os.symlink("../top/top.runs/impl_1/d_1_wrapper.bit", str(out_dir / "qick_216.bit"))
os.symlink("../top/top.gen/sources_1/bd/d_1/hw_handoff/d_1.hwh", str(out_dir / "qick_216.hwh"))

print("Native Linux symlinks successfully recreated!")

Native Linux symlinks successfully recreated!


In [ ]:
from qick import QickSoc

# Load the firmware bitstream
soc = QickSoc("/home/xilinx/jupyter_notebooks/qick/firmware/projects/qick_tprocv2_216_16mixmux16/out/qick_216.bit", force_init_clks=True)

soccfg = soc

# Print to check soc details and show code has run
print(soc)

configuring reference clock chips, as requested
LMK04208 clock reference = 122.880 MHz, LMX2594 clock synth = 204.800 MHz
QICK running on ZCU111, software version 0.2.418

Firmware configuration (built Wed Jun 10 15:03:08 2026):

	Global clocks (MHz): tProc dispatcher timing 384.000, RF reference 204.800
	Groups of related clocks: [tProc core clock, tProc timing clock, DAC tile 0, DAC tile 1], [ADC tile 0, ADC tile 2]

	8 signal generator channels:
	0:	axis_sg_mixmux8_v1 - fs=6144.000 Msps, fabric=384.000 MHz
		32-bit DDS, range=1536.000 MHz
		DAC tile 0, blk 0 is DAC228_T0_CH0, or RF board DAC port 0
	1:	axis_sg_mixmux8_v1 - fs=6144.000 Msps, fabric=384.000 MHz
		32-bit DDS, range=1536.000 MHz
		DAC tile 0, blk 1 is DAC228_T0_CH1, or RF board DAC port 1
	2:	axis_sg_mixmux8_v1 - fs=6144.000 Msps, fabric=384.000 MHz
		32-bit DDS, range=1536.000 MHz
		DAC tile 0, blk 2 is DAC228_T0_CH2, or RF board DAC port 2
	3:	axis_sg_mixmux8_v1 - fs=6144.000 Msps, fabric=384.000 MHz
		32-bit DDS, ran

In [2]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from qick import *
from qick.asm_v2 import AveragerProgramV2 

# useful libraries
import numpy as np
import random 
import time
import math

from scipy.signal import windows
from collections import defaultdict

In [ ]:
soc.reset_gens()

# Shaped Schedule, Main Code

In [3]:
RO_CH = 0
DEFAULT_STEP_US   = 2.0    # staircase step target (µs) when a tone doesn't set shape_step_us
MIN_STEP_US       = 1.0    # hard floor on staircase step length
MAX_STEP_US       = 10.0   # hard ceiling on requested step_us
GAIN_MERGE_TOL    = 1e-3   # consecutive samples within this gain difference get merged into one step
GAIN_EPS          = 1e-3   # steps below this gain are dropped entirely
MAX_INSTRUCTIONS  = 8192   # tProc program memory limit -- verify against your actual build

In [ ]:
def decompose_schedule(pulse_schedule):
    """Slices per-channel schedule into non-overlapping time segments with active-tone masks."""

    ch_pulses = defaultdict(list)
    for tone in pulse_schedule:
        ch_pulses[tone["channel"]].append(tone)

    decomposed = []
    for ch, tones in ch_pulses.items():
        time_points = set()
        for tone in tones:
            time_points.add(tone["t_off"])
            time_points.add(tone["t_off"] + tone["pulse_length"])
        time_points = sorted(time_points)

        for i in range(len(time_points) - 1):
            t_start, t_end = time_points[i], time_points[i + 1]
            active = [
                tone for tone in tones
                if tone["t_off"] <= t_start and tone["t_off"] + tone["pulse_length"] >= t_end
            ]
            if active:
                decomposed.append({
                    "channel": ch, "t_off": t_start,
                    "pulse_length": t_end - t_start, "active_tones": active,
                })

    decomposed.sort(key=lambda p: (p["channel"], p["t_off"]))
    return decomposed

In [ ]:
def tukey_window(n, alpha):
    """Hand-rolled Tukey window (no scipy dependency). alpha=0 -> rect, alpha=1 -> Hann.

    Args:
        n: number of raw samples (pre-merge staircase levels)
        alpha: taper fraction

    Returns:
        w: np.ndarray of length n, gain multipliers in [0, 1]
    """

    if n <= 1:
        return np.ones(max(n, 1))
    if alpha <= 0:
        return np.ones(n)
    if alpha >= 1:
        return 0.5 * (1 - np.cos(2 * np.pi * np.arange(n) / (n - 1)))

    w = np.ones(n)
    edge = int(np.floor(alpha * (n - 1) / 2.0))
    x = np.arange(0, edge + 1)
    taper = 0.5 * (1 + np.cos(np.pi * (2 * x / (alpha * (n - 1)) - 1)))
    w[:edge + 1] = taper
    w[n - edge - 1:] = taper[::-1]
    return w


def expand_tone_to_staircase(tone, default_step_us, min_levels=4,
                              gain_eps=GAIN_EPS, gain_merge_tol=GAIN_MERGE_TOL):
    """Splits one tone into flat const-gain sub-tones tracing a Tukey envelope; const tones pass through unchanged.

    Consecutive raw samples with near-equal gain (e.g. a flat top) are merged
    into one longer step -- this is what keeps instruction count low.

    Args:
        tone: schedule entry. If style="tukey", also reads alpha (default 0.2)
            and optionally shape_step_us (overrides default_step_us).
        default_step_us: the resolved fallback step target (µs) used when a
            tone doesn't set its own "shape_step_us". This is REQUIRED and has
            no built-in default -- callers must resolve it from cfg["shape_step_us"]
            (see ShapedMixMuxProg._initialize) so the step size is always an
            explicit, visible choice rather than a hidden module constant.
        min_levels: minimum RAW samples pre-merge (controls taper resolution,
            not final step count).
        gain_eps: drop steps below this gain.
        gain_merge_tol: max gain delta between adjacent raw samples to merge.

    Returns:
        sub_tones: list of sub-tone dicts, one per merged step.
    """

    style = tone.get("style", "const")
    if style == "const":
        return [dict(tone)]
    if style != "tukey":
        raise ValueError(f"Unknown pulse style '{style}' on tone {tone}")

    # Per-tone override still wins over the (now-required) schedule-level default.
    step_us = tone.get("shape_step_us", default_step_us)
    if step_us < MIN_STEP_US:
        raise ValueError(f"Tone {tone.get('channel')}/{tone.get('freq')}: shape_step_us={step_us} below {MIN_STEP_US}us floor.")
    if step_us > MAX_STEP_US:
        raise ValueError(f"Tone {tone.get('channel')}/{tone.get('freq')}: shape_step_us={step_us} above {MAX_STEP_US}us ceiling.")

    n_levels = max(min_levels, int(round(tone["pulse_length"] / step_us)))

    max_levels_for_floor = int(tone["pulse_length"] // MIN_STEP_US)
    if max_levels_for_floor < 1:
        raise ValueError(f"Tone {tone.get('channel')}/{tone.get('freq')}: pulse_length too short for even one {MIN_STEP_US}us step.")
    n_levels = min(n_levels, max_levels_for_floor)

    window = tukey_window(n_levels, tone.get("alpha", 0.2))
    raw_gains = tone["gain"] * window
    step_len = tone["pulse_length"] / n_levels

    sub_tones = []
    run_start_i = 0
    run_gain = raw_gains[0]
    for i in range(1, n_levels + 1):
        ended = (i == n_levels) or (abs(raw_gains[i] - run_gain) > gain_merge_tol)
        if ended:
            run_len_samples = i - run_start_i
            if run_gain >= gain_eps:
                sub_tones.append({
                    "channel": tone["channel"], "freq": tone["freq"], "gain": run_gain,
                    "phase": tone["phase"], "pulse_length": run_len_samples * step_len,
                    "t_off": tone["t_off"] + run_start_i * step_len,
                })
            if i < n_levels:
                run_start_i = i
                run_gain = raw_gains[i]

    return sub_tones


def expand_pulse_schedule(pulse_schedule, default_step_us):
    """Expands every tone in the schedule; output feeds directly into decompose_schedule.

    Args:
        pulse_schedule: list of tone dicts.
        default_step_us: required fallback step (µs) for tones that don't set
            their own "shape_step_us". Callers should pass cfg["shape_step_us"]
            here -- there's no implicit default baked into this function.
    """

    expanded = []
    for tone in pulse_schedule:
        expanded.extend(expand_tone_to_staircase(tone, default_step_us))
    return expanded

In [ ]:
# Main class for mixmux prog
class ShapedMixMuxProg(AveragerProgramV2):
    def _initialize(self, cfg):
        # step_us hierarchy is now just two levels:
        #   1. cfg["shape_step_us"]        -- required, schedule-wide default
        #   2. tone["shape_step_us"]       -- optional, per-tone override
        # The old three-level chain (module DEFAULT_STEP_US -> cfg -> tone)
        # is gone; if you forget to set it in cfg, this fails loudly instead
        # of silently picking a step size you never chose.
        if "shape_step_us" not in cfg:
            raise ValueError(
                "[ShapedMixMuxProg] cfg['shape_step_us'] is required (global "
                "default staircase step in µs, e.g. 2.0). Individual tones can "
                "still override it by setting their own 'shape_step_us'."
            )
        step_us = cfg["shape_step_us"]

        expanded_schedule = expand_pulse_schedule(cfg["pulse_schedule"], step_us)

        ch_tones = defaultdict(dict)
        for tone in expanded_schedule:
            key = (round(tone["freq"], 6), round(tone["gain"], 6))
            ch_tones[tone["channel"]].setdefault(key, tone["phase"])

        self.ch_freq_index = {}
        for ch, tones in ch_tones.items():
            keys   = list(tones.keys())
            freqs  = [k[0] for k in keys]
            gains  = [k[1] for k in keys]
            phases = [tones[k] for k in keys]
            self.ch_freq_index[ch] = {k: i for i, k in enumerate(keys)}

            self.declare_gen(
                ch=ch, nqz=1, ro_ch=cfg["ro_ch"],
                mixer_freq=0,
                mux_freqs=freqs,
                mux_gains=gains,
                mux_phases=phases
            )

        self.declare_readout(ch=cfg["ro_ch"], length=cfg["ro_len"])
        first = cfg["pulse_schedule"][0]
        self.add_readoutconfig(
            ch=cfg["ro_ch"], name="myro",
            freq=first["freq"], phase=0,
            gen_ch=first["channel"]
        )

        self.decomposed = decompose_schedule(expanded_schedule)

        if len(self.decomposed) > MAX_INSTRUCTIONS:
            raise ValueError(
                f"[ShapedMixMuxProg] {len(self.decomposed)} pulse instructions "
                f"generated (step_us={step_us}), exceeds the {MAX_INSTRUCTIONS} limit. "
                f"Increase shape_step_us, raise gain_merge_tol, or reduce alpha."
            )
        print(f"[ShapedMixMuxProg] {len(self.decomposed)} pulse instructions generated (step_us={step_us})")

        for i, seg in enumerate(self.decomposed):
            ch = seg["channel"]
            keys = [(round(t["freq"], 6), round(t["gain"], 6)) for t in seg["active_tones"]]

            if len(keys) > 16:  # mixmux16 simultaneous-tone limit
                raise ValueError(f"Channel {ch}, t={seg['t_off']}: {len(keys)} tones active, exceeds 16-tone limit.")

            mask = [self.ch_freq_index[ch][k] for k in keys]
            self.add_pulse(
                ch=ch,
                name=f"pulse_{i}",
                style="const",
                length=seg["pulse_length"],
                mask=mask
            )

    def _body(self, cfg):
        self.trigger(ros=[cfg["ro_ch"]], pins=[0], t=cfg["trig_time"])
        for i, seg in enumerate(self.decomposed):
            self.pulse(ch=seg["channel"], name=f"pulse_{i}", t=seg["t_off"])

In [ ]:
config = {
    "ro_ch":     0,
    "ro_len":    1.9,
    "trig_time": 0.4,
    "shape_step_us": 1.0,  # required global default staircase step; override per-tone via "shape_step_us"

    "pulse_schedule": [
        {"channel": 0, "freq": 0,   "gain": 1.0, "phase": 0, "pulse_length": 800.0, "t_off": 100.0},

        # Tukey-shaped tone -- each tone shapes independently of the others
        {"channel": 1, "freq": 200, "gain": 1.0, "phase": 0, "pulse_length": 800.0,
         "t_off": 100.0, "style": "tukey", "alpha": 0.3},

        # a second, simultaneous shaped tone, same channel, different alpha/step
        {"channel": 1, "freq": 225, "gain": 0.8, "phase": 0, "pulse_length": 800.0,
         "t_off": 100.0, "style": "tukey", "alpha": 0.5, "shape_step_us": 1.0},

        # plain unshaped tone layered in on the same channel, unaffected
        {"channel": 1, "freq": 250, "gain": 1.0, "phase": 0, "pulse_length": 150.0, "t_off": 100.0},
    ],
}

prog = ShapedMixMuxProg(soccfg, reps=100, final_delay=0.0, cfg=config)

NameError: name 'soccfg' is not defined

# Example Plotting & Testing

For testing basics, or getting an idea of how the pulse looks, without running

In [2]:
# Example Plotting & Testing

# For testing basics, or getting an idea of how the pulse looks, without running
def plot_tukey_staircase(tone, default_step_us, min_levels=4, gain_eps=1e-3):
    """
    Plots a shaped tone's staircase gain profile against the ideal
    continuous Tukey envelope, for sanity-checking.

    Args:
        tone: a single schedule entry dict, same format as used in
            pulse_schedule.
        default_step_us: required -- pass the same step (µs) you'd use for
            this tone in the real schedule (e.g. config["shape_step_us"],
            or the tone's own "shape_step_us" if it sets one).
        min_levels, gain_eps: passed straight through to
            expand_tone_to_staircase -- see that function's docstring.
    """

    steps = expand_tone_to_staircase(tone, default_step_us, min_levels, gain_eps)

    # Build the staircase trace: repeat each step's gain across its start/end time
    # so plt.plot draws flat horizontal segments instead of connecting midpoints
    stair_t, stair_g = [], []
    for s in steps:
        stair_t += [s["t_off"], s["t_off"] + s["pulse_length"]]
        stair_g += [s["gain"], s["gain"]]

    # Ideal continuous envelope for comparison, sampled finely regardless of step count
    t_ideal = np.linspace(tone["t_off"], tone["t_off"] + tone["pulse_length"], 1000)
    n_ideal = 1000
    window_ideal = tukey_window(n_ideal, tone.get("alpha", 0.2))
    gain_ideal = tone["gain"] * window_ideal

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(t_ideal, gain_ideal, "--", alpha=0.5, label="ideal continuous envelope")
    ax.plot(stair_t, stair_g, "-", linewidth=2, label=f"staircase ({len(steps)} steps)")
    ax.set_xlabel("time (µs)")
    ax.set_ylabel("gain")
    ax.set_title(f"ch={tone['channel']}, freq={tone['freq']}MHz, alpha={tone.get('alpha', 0.2)}")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.show()

    print(f"{len(steps)} steps, {steps[0]['pulse_length']:.3f}us each "
          f"(requested step_us={tone.get('shape_step_us', default_step_us)})")

In [3]:
# Note a max instruction count of 8192

example_tone = {
    "channel": 1, "freq": 200, "gain": 1.0, "phase": 0,
    "pulse_length": 800.0, "t_off": 100.0,
    "style": "tukey", "alpha": 0.3,
}

plot_tukey_staircase(example_tone, default_step_us=2.0)

merged_steps = expand_tone_to_staircase(example_tone, default_step_us=2.0)
n_steps = len(merged_steps)

# Instruction estimate = number of steps

print(f"{n_steps} merged staircase steps (post flat-top merge), same as instruction number")

NameError: name 'expand_tone_to_staircase' is not defined

Estimate the instruction count of an entire pulse sequence to see if it's under the limit

In [ ]:
def estimate_instruction_count(pulse_schedule, default_step_us, min_levels=4,
                                gain_eps=GAIN_EPS, gain_merge_tol=GAIN_MERGE_TOL,
                                max_simultaneous_tones=16, verbose=True):
    """
    Estimate the instruction count of an entire pulse sequence to see if it's
    under the limit.

    Runs the real expand -> decompose pipeline (no hardware calls) and counts
    the result, since decompose_schedule's time-boundary unioning makes a
    per-tone estimate unreliable when shaped tones overlap.

    instruction_estimate == segment_count

    Args:
        pulse_schedule: list of tone dicts.
        default_step_us: required -- pass the same value you'll use for
            cfg["shape_step_us"] so the estimate matches the real run.

    Returns:
        dict with expanded_tone_count, segment_count, instruction_estimate,
        per_channel, max_simultaneous, over_simultaneous_limit.
    """
    expanded = expand_pulse_schedule(pulse_schedule, default_step_us)
    decomposed = decompose_schedule(expanded)

    per_channel = defaultdict(int)
    max_simultaneous = defaultdict(int)
    over_limit = []

    for seg in decomposed:
        ch = seg["channel"]
        per_channel[ch] += 1
        n_active = len(seg["active_tones"])
        max_simultaneous[ch] = max(max_simultaneous[ch], n_active)
        if n_active > max_simultaneous_tones:
            over_limit.append((ch, seg["t_off"], n_active))

    result = {
        "expanded_tone_count": len(expanded),
        "segment_count": len(decomposed),
        "instruction_estimate": len(decomposed),
        "per_channel": dict(per_channel),
        "max_simultaneous": dict(max_simultaneous),
        "over_simultaneous_limit": over_limit,
    }

    if verbose:
        print(f"Expanded tones (post-merge): {result['expanded_tone_count']}")
        print(f"Segments / instruction estimate: {result['segment_count']}")
        for ch in sorted(per_channel):
            print(f"  ch={ch}: {per_channel[ch]} segments, max {max_simultaneous[ch]} simultaneous tones")
        if over_limit:
            print(f"WARNING: {len(over_limit)} segment(s) exceed the {max_simultaneous_tones}-tone limit:")
            for ch, t_off, n in over_limit:
                print(f"  ch={ch}, t={t_off}: {n} tones active")

    return result

In [ ]:
stats = estimate_instruction_count(config["pulse_schedule"], default_step_us=config["shape_step_us"])

if stats["instruction_estimate"] >= 8192:  # placeholder ceiling -- verify real tProc program memory
    print("Likely over instruction budget -- increase default_step_us, "
          "raise gain_merge_tol, reduce alpha on some tones, etc")

Expanded tones (post-merge): 604
Segments / instruction estimate: 383
  ch=0: 1 segments, max 1 simultaneous tones
  ch=1: 382 segments, max 3 simultaneous tones


#